In [1]:
# # Evandro Ribeiro Gomes Coelho
# # TCCII

# packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn.metrics as m

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.datasets import make_regression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler 
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

In [2]:
# # import data

fl_path = 'data/'
fl_ext = '.csv'

fl_name = 'df_mlp'
df = pd.DataFrame(pd.read_csv(fl_path + fl_name + fl_ext, sep=',', dtype={'Holiday Flag':'int'}))

del fl_path, fl_ext, fl_name

In [3]:
df = df.set_index('Timestamp')

In [4]:
df = df.rename(columns={'Value':'Energia',
                          'Temperature':'Temperatura',
                          'Holiday Flag':'Marcação Feriado',
                          'hour':'Hora',
                          'min':'Minuto',
                          'time':'Tempo',
                          'hr_sin':'Hora Seno',
                          'hr_cos':'Hora Cosseno',
                          'Value Prev':'Energia Anterior',
                          'Temperature Prev':'Temperatura Anterior'})
df

,Energia,Temperatura,Marcação Feriado,Hora,Minuto,Tempo,Hora Seno,Hora Cosseno
Timestamp,,,,,,,,
2015-07-03 14:30:00+00:00,6997.984367,31.0,0,14,30,14.5,-0.608761,-0.793353
2015-07-03 15:30:00+00:00,15518.012898,33.0,0,15,30,15.5,-0.793353,-0.608761
2015-07-03 16:30:00+00:00,6674.491991,34.0,0,16,30,16.5,-0.923880,-0.382683
2015-07-03 17:30:00+00:00,4920.282616,33.0,0,17,30,17.5,-0.991445,-0.130526
2015-07-03 18:30:00+00:00,4118.787012,34.0,0,18,30,18.5,-0.991445,0.130526
...,...,...,...,...,...,...,...,...
2017-11-08 09:30:00+00:00,9823.167728,6.0,0,9,30,9.5,0.608761,-0.793353
2017-11-08 10:30:00+00:00,9813.048374,6.0,0,10,30,10.5,0.382683,-0.923880
2017-11-08 11:30:00+00:00,11714.319199,7.0,0,11,30,11.5,0.130526,-0.991445


In [5]:
# creating features
df['Energia Anterior'] = df.Energia.shift(1)
df['Temperatura Anterior'] = df.Temperatura.shift(1)
df = df.fillna(0)
df

,Energia,Temperatura,Marcação Feriado,Hora,Minuto,Tempo,Hora Seno,Hora Cosseno,Energia Anterior,Temperatura Anterior
Timestamp,,,,,,,,,,
2015-07-03 14:30:00+00:00,6997.984367,31.0,0,14,30,14.5,-0.608761,-0.793353,0.000000,0.0
2015-07-03 15:30:00+00:00,15518.012898,33.0,0,15,30,15.5,-0.793353,-0.608761,6997.984367,31.0
2015-07-03 16:30:00+00:00,6674.491991,34.0,0,16,30,16.5,-0.923880,-0.382683,15518.012898,33.0
2015-07-03 17:30:00+00:00,4920.282616,33.0,0,17,30,17.5,-0.991445,-0.130526,6674.491991,34.0
2015-07-03 18:30:00+00:00,4118.787012,34.0,0,18,30,18.5,-0.991445,0.130526,4920.282616,33.0
...,...,...,...,...,...,...,...,...,...,...
2017-11-08 09:30:00+00:00,9823.167728,6.0,0,9,30,9.5,0.608761,-0.793353,9907.625408,5.0
2017-11-08 10:30:00+00:00,9813.048374,6.0,0,10,30,10.5,0.382683,-0.923880,9823.167728,6.0
2017-11-08 11:30:00+00:00,11714.319199,7.0,0,11,30,11.5,0.130526,-0.991445,9813.048374,6.0


In [6]:
model = Pipeline([
    ('scaler', 
         MinMaxScaler(
            feature_range=(-1, 1), copy=True
        )
    ),
    ('regressor',
        MLPRegressor(
            hidden_layer_sizes=(43,),  activation='tanh', solver='adam', alpha=0.001, batch_size='auto',
            learning_rate='constant', learning_rate_init=0.01, power_t=0.5, max_iter=1500, shuffle=True,
            random_state=0, tol=0.0001, verbose=False, warm_start=False, momentum=0, nesterovs_momentum=True,
            early_stopping=False, validation_fraction=0.1, beta_1=0.9, beta_2=0.999, epsilon=1e-08, n_iter_no_change=10, max_fun=3500
        )
    )
])

parameters = {
    'regressor__hidden_layer_sizes':[(10,), (30,), (50,)],
    'regressor__max_iter':[1500, 3500, 5000]
}

grid = GridSearchCV(
    model,
    parameters,
    scoring='r2',
    n_jobs = -1,
    cv = 10,
    verbose=True
)

In [7]:
# alter features for the tests
features = ['Temperatura', 'Temperatura Anterior', 'Marcação Feriado', 'Energia Anterior', 'Hora Seno', 'Hora Cosseno']
X = df[features].values
y = df['Energia'].values

In [8]:
# spliting data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 1)

In [9]:
# gridsearch test
grid.fit(X_train, y_train)

Fitting 10 folds for each of 9 candidates, totalling 90 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  6.9min
[Parallel(n_jobs=-1)]: Done  90 out of  90 | elapsed: 21.1min finished


GridSearchCV(cv=10,
             estimator=Pipeline(steps=[('scaler',
                                        MinMaxScaler(feature_range=(-1, 1))),
                                       ('regressor',
                                        MLPRegressor(activation='tanh',
                                                     alpha=0.001,
                                                     hidden_layer_sizes=(43,),
                                                     learning_rate_init=0.01,
                                                     max_fun=3500,
                                                     max_iter=1500, momentum=0,
                                                     random_state=0))]),
             n_jobs=-1,
             param_grid={'regressor__hidden_layer_sizes': [(10,), (30,), (50,)],
                         'regressor__max_iter': [1500, 3500, 5000]},
             scoring='r2', verbose=True)

In [10]:
frame = pd.DataFrame(grid.cv_results_)
frame.to_csv('measures/frame.csv', index = False, header=True)
frame

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_regressor__hidden_layer_sizes,param_regressor__max_iter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,split5_test_score,split6_test_score,split7_test_score,split8_test_score,split9_test_score,mean_test_score,std_test_score,rank_test_score
0,70.321631,2.129499,0.001589,0.001275,"(10,)",1500,"{'regressor__hidden_layer_sizes': (10,), 'regr...",-0.000190,-0.002251,-0.003918,-0.000411,-0.001148,-0.000799,-0.000192,-0.000007,-0.001484,-0.000039,-0.001044,0.001180,7
1,72.119385,2.021942,0.002330,0.004853,"(10,)",3500,"{'regressor__hidden_layer_sizes': (10,), 'regr...",-0.000190,-0.002251,-0.003918,-0.000411,-0.001148,-0.000799,-0.000192,-0.000007,-0.001484,-0.000039,-0.001044,0.001180,7
2,68.189216,0.819519,0.007413,0.006526,"(10,)",5000,"{'regressor__hidden_layer_sizes': (10,), 'regr...",-0.000190,-0.002251,-0.003918,-0.000411,-0.001148,-0.000799,-0.000192,-0.000007,-0.001484,-0.000039,-0.001044,0.001180,7
3,135.194353,6.953863,0.008422,0.008258,"(30,)",1500,"{'regressor__hidden_layer_sizes': (30,), 'regr...",0.852334,0.851384,0.864713,0.859807,0.879623,0.878483,0.851131,0.870489,0.868469,0.853397,0.862983,0.010473,6
4,135.909498,7.535977,0.006455,0.007718,"(30,)",3500,"{'regressor__hidden_layer_sizes': (30,), 'regr...",0.852334,0.851384,0.864713,0.859807,0.879623,0.878483,0.852124,0.870489,0.868469,0.853397,0.863082,0.010365,4
5,134.585425,7.486957,0.004910,0.007276,"(30,)",5000,"{'regressor__hidden_layer_sizes': (30,), 'regr...",0.852334,0.851384,0.864713,0.859807,0.879623,0.878483,0.852124,0.870489,0.868469,0.853397,0.863082,0.010365,4
6,119.758562,8.778664,0.001981,0.004968,"(50,)",1500,"{'regressor__hidden_layer_sizes': (50,), 'regr...",0.859349,0.858369,0.870735,0.866293,0.887214,0.881356,0.859647,0.878222,0.875569,0.857103,0.869386,0.010279,1
7,120.803499,8.176283,0.004909,0.007293,"(50,)",3500,"{'regressor__hidden_layer_sizes': (50,), 'regr...",0.859349,0.858369,0.870735,0.866293,0.887214,0.881356,0.859647,0.878222,0.875569,0.857103,0.869386,0.010279,1
8,115.221912,7.957712,0.006485,0.007951,"(50,)",5000,"{'regressor__hidden_layer_sizes': (50,), 'regr...",0.859349,0.858369,0.870735,0.866293,0.887214,0.881356,0.859647,0.878222,0.875569,0.857103,0.869386,0.010279,1
